# Solstrom HPC Results Dashboard

Run this notebook after syncing Solstrom output with `sync_from_solstorm.sh`. It inspects the synced `master_summary.csv`, nested `performance.csv` files, and saved NumPy arrays without requiring widgets or project imports.

In [1]:
from pathlib import Path
import ast
import os
import re
import warnings

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

# Configure this if sync_from_solstorm.sh writes artifacts somewhere else.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

ARTIFACTS_ROOT = PROJECT_ROOT / "artifacts" / "hpc_runs"
MASTER_SUMMARY_PATH = ARTIFACTS_ROOT / "master_summary.csv"

# Optional selectors for the quick array preview near the bottom.
SELECTED_RUN_NAME = None      # Example: "macro_fwd_ann_fwd3_macro32_100runs_top10"
SELECTED_RUN_DIR = None       # Example: ARTIFACTS_ROOT / "some_run" / "20260427_120000"
SELECTED_MATURITY = None      # Example: 12, "12", or None to use the first column
MAX_PREVIEW_POINTS = 300

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)
pd.set_option("display.width", 160)

print(f"Project root: {PROJECT_ROOT}")
print(f"Artifacts root: {ARTIFACTS_ROOT}")
print(f"Master summary: {MASTER_SUMMARY_PATH}")

Project root: /Users/OlavNikolai/TIO4900-Replication
Artifacts root: /Users/OlavNikolai/TIO4900-Replication/artifacts/hpc_runs
Master summary: /Users/OlavNikolai/TIO4900-Replication/artifacts/hpc_runs/master_summary.csv


In [3]:
def note(message, level="info"):
    prefix = {"info": "Info", "warn": "Warning", "error": "Error"}.get(level, "Info")
    display(Markdown(f"**{prefix}:** {message}"))


def read_csv(path, **kwargs):
    path = Path(path)
    if not path.exists():
        return pd.DataFrame()
    try:
        return pd.read_csv(path, **kwargs)
    except Exception as exc:
        note(f"Could not read `{path}`: `{type(exc).__name__}: {exc}`", "error")
        return pd.DataFrame()


def first_col(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    lowered = {str(c).lower(): c for c in df.columns}
    for col in candidates:
        if col.lower() in lowered:
            return lowered[col.lower()]
    return None


def coerce_datetime(series):
    return pd.to_datetime(series, errors="coerce")


def infer_family(run_name):
    text = str(run_name).lower()
    if "group" in text or "grp" in text:
        return "group_ensemble_ann"
    if "macro" in text and "fwd" in text:
        return "macro_forward_ann"
    if "fwd_ann" in text or text.startswith("fwd"):
        return "forward_ann"
    if "pca" in text:
        return "pca"
    if "pls" in text:
        return "pls"
    if "lasso" in text:
        return "lasso"
    return "unknown"


def parse_shape(value):
    if pd.isna(value):
        return None
    if isinstance(value, (tuple, list)):
        return tuple(value)
    try:
        parsed = ast.literal_eval(str(value))
        if isinstance(parsed, (tuple, list)):
            return tuple(parsed)
    except Exception:
        return None
    return None


def sort_by_latest(df):
    if df.empty:
        return df
    out = df.copy()
    timestamp_col = first_col(out, ["timestamp", "created_at", "datetime", "date"])
    if timestamp_col:
        out["_sort_timestamp"] = coerce_datetime(out[timestamp_col])
    else:
        out["_sort_timestamp"] = pd.NaT
    if "run_dir" in out.columns:
        out["_sort_run_dir"] = out["run_dir"].astype(str)
    else:
        out["_sort_run_dir"] = ""
    return out.sort_values(["_sort_timestamp", "_sort_run_dir"], ascending=[False, False], na_position="last")


def latest_existing_run_dirs(summary_df):
    if summary_df.empty or "run_dir" not in summary_df.columns:
        return []
    run_dirs = []
    for raw in sort_by_latest(summary_df)["run_dir"].dropna().astype(str):
        if not raw.strip():
            continue
        path = Path(raw)
        if path.exists():
            run_dirs.append(path)
        else:
            # Solstrom absolute paths may differ locally after rsync; fall back to matching tail under ARTIFACTS_ROOT.
            parts = path.parts[-2:]
            if len(parts) == 2:
                matches = list(ARTIFACTS_ROOT.glob(f"**/{parts[0]}/{parts[1]}"))
                run_dirs.extend([m for m in matches if m.is_dir()])
    seen = set()
    unique = []
    for path in run_dirs:
        key = str(path.resolve())
        if key not in seen:
            unique.append(path)
            seen.add(key)
    return unique

## Master Summary

Loads `artifacts/hpc_runs/master_summary.csv` by default and separates completed and failed rows.

In [5]:
master_summary = read_csv(MASTER_SUMMARY_PATH)

if master_summary.empty:
    note(f"No master summary found yet at `{MASTER_SUMMARY_PATH}`. Run `sync_from_solstorm.sh` first, or change `ARTIFACTS_ROOT` above.", "warn")
else:
    status_col = first_col(master_summary, ["status", "state", "result"])
    timestamp_col = first_col(master_summary, ["timestamp", "created_at", "datetime", "date"])
    if timestamp_col:
        master_summary[timestamp_col] = coerce_datetime(master_summary[timestamp_col])

    display(master_summary)

    if status_col:
        status = master_summary[status_col].astype(str).str.lower()
        completed = master_summary[status.isin(["ok", "completed", "complete", "success", "succeeded"])]
        failed = master_summary[status.str.contains("fail|error|exception", regex=True, na=False)]

        note(f"Completed rows: {len(completed)}. Failed rows: {len(failed)}. Total rows: {len(master_summary)}.")
        if not completed.empty:
            display(Markdown("### Completed"))
            display(sort_by_latest(completed).drop(columns=[c for c in ["_sort_timestamp", "_sort_run_dir"] if c in completed.columns], errors="ignore"))
        if not failed.empty:
            display(Markdown("### Failed"))
            display(sort_by_latest(failed).drop(columns=[c for c in ["_sort_timestamp", "_sort_run_dir"] if c in failed.columns], errors="ignore"))
    else:
        note("No `status` column was found in the master summary; displaying all rows only.", "warn")

**Warning:** No master summary found yet at `/Users/OlavNikolai/TIO4900-Replication/artifacts/hpc_runs/master_summary.csv`. Run `sync_from_solstorm.sh` first, or change `ARTIFACTS_ROOT` above.

## Performance Files

Recursively finds nested `performance.csv` files, combines them, and attaches run metadata inferred from the directory layout and master summary.

In [ ]:
performance_paths = sorted(ARTIFACTS_ROOT.rglob("performance.csv")) if ARTIFACTS_ROOT.exists() else []

if not performance_paths:
    note(f"No nested `performance.csv` files found under `{ARTIFACTS_ROOT}` yet.", "warn")
    performance = pd.DataFrame()
else:
    frames = []
    for path in performance_paths:
        df = read_csv(path)
        if df.empty:
            continue
        run_dir = path.parent
        df = df.copy()
        df["performance_path"] = str(path)
        df["run_dir"] = str(run_dir)
        df["run_timestamp"] = run_dir.name
        df["run_name"] = run_dir.parent.name if run_dir.parent != ARTIFACTS_ROOT else run_dir.name
        df["family"] = df["run_name"].map(infer_family)
        frames.append(df)

    performance = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

    if not performance.empty and not master_summary.empty and "run_dir" in master_summary.columns:
        summary_cols = [c for c in ["run_dir", "timestamp", "status", "num_checkpoints", "total_checkpoint_gb", "forecasts_arr_shape", "losses_arr_shape", "error"] if c in master_summary.columns]
        summary_meta = master_summary[summary_cols].drop_duplicates("run_dir", keep="last")
        performance = performance.merge(summary_meta, on="run_dir", how="left", suffixes=("", "_summary"))

    note(f"Found {len(performance_paths)} performance files and loaded {len(performance)} performance rows.")
    display(performance)

## R2 Overview

Shows out-of-sample R2 by model, inferred model family, and maturity. Plots are shown when `matplotlib` is installed.

In [ ]:
if performance.empty:
    note("No performance data to summarize yet.", "warn")
else:
    r2_col = first_col(performance, ["r2_oos", "r2", "oos_r2"])
    maturity_col = first_col(performance, ["maturity", "horizon", "tenor"])
    method_col = first_col(performance, ["ensemble_method", "selection_metric", "method"])
    if not r2_col or not maturity_col:
        note(f"Could not find R2/maturity columns. Available columns: {list(performance.columns)}", "error")
    else:
        perf = performance.copy()
        perf[r2_col] = pd.to_numeric(perf[r2_col], errors="coerce")
        perf[maturity_col] = perf[maturity_col].astype(str)
        if method_col:
            perf[method_col] = perf[method_col].fillna("single_forecast").astype(str)
            model_index = ["family", "run_name", method_col]
        else:
            model_index = ["family", "run_name"]

        r2_table = (
            perf.pivot_table(index=model_index, columns=maturity_col, values=r2_col, aggfunc="mean")
            .sort_index()
        )
        display(Markdown("### R2 by Family, Model, Ensemble Method, and Maturity"))
        display(r2_table)

        family_index = ["family", method_col] if method_col else "family"
        family_table = perf.pivot_table(index=family_index, columns=maturity_col, values=r2_col, aggfunc="mean").sort_index()
        display(Markdown("### Average R2 by Family and Ensemble Method"))
        display(family_table)

        group_cols = [maturity_col, method_col] if method_col else [maturity_col]
        best_rows = perf.sort_values(r2_col, ascending=False).groupby(group_cols, as_index=False).head(5)
        display(Markdown("### Top Rows per Maturity and Ensemble Method"))
        display(best_rows[[c for c in ["maturity", "ensemble_method", "family", "run_name", r2_col, "rsz_pval", "run_dir"] if c in best_rows.columns]])

        try:
            import matplotlib.pyplot as plt

            latest_perf = perf.copy()
            latest_perf["_run_dt"] = pd.to_datetime(latest_perf.get("run_timestamp"), format="%Y%m%d_%H%M%S", errors="coerce")
            dedupe_cols = ["run_name", maturity_col] + ([method_col] if method_col else [])
            latest_perf = latest_perf.sort_values("_run_dt").drop_duplicates(dedupe_cols, keep="last")

            bar_index = ["run_name", method_col] if method_col else "run_name"
            bar_data = latest_perf.groupby(bar_index, dropna=False)[r2_col].mean().sort_values(ascending=False)
            if not bar_data.empty:
                height = max(4, min(16, 0.35 * len(bar_data)))
                fig, ax = plt.subplots(figsize=(10, height))
                bar_data.sort_values().plot(kind="barh", ax=ax, color="#3b82f6")
                ax.set_title("Mean R2 by latest model run")
                ax.set_xlabel("Mean out-of-sample R2")
                ax.set_ylabel("Run / ensemble method")
                ax.axvline(0, color="black", linewidth=0.8)
                fig.tight_layout()
                plt.show()

            heat_index = ["run_name", method_col] if method_col else "run_name"
            heat = latest_perf.pivot_table(index=heat_index, columns=maturity_col, values=r2_col, aggfunc="mean")
            if not heat.empty:
                fig, ax = plt.subplots(figsize=(max(7, 0.8 * len(heat.columns)), max(4, 0.35 * len(heat))))
                im = ax.imshow(heat.fillna(0), aspect="auto", cmap="RdYlGn")
                ax.set_xticks(range(len(heat.columns)))
                ax.set_xticklabels(heat.columns)
                ax.set_yticks(range(len(heat.index)))
                ax.set_yticklabels([" | ".join(map(str, item)) if isinstance(item, tuple) else str(item) for item in heat.index])
                ax.set_title("Latest R2 heatmap")
                fig.colorbar(im, ax=ax, label="R2")
                fig.tight_layout()
                plt.show()
        except Exception as exc:
            note(f"Plotting skipped: `{type(exc).__name__}: {exc}`", "warn")


## Latest Run Paths

Lists latest synced run directories with key artifact availability.

In [ ]:
if not ARTIFACTS_ROOT.exists():
    note(f"Artifacts root `{ARTIFACTS_ROOT}` does not exist yet.", "warn")
    latest_runs = pd.DataFrame()
else:
    run_dirs = latest_existing_run_dirs(master_summary)
    if not run_dirs:
        run_dirs = sorted({p.parent for p in performance_paths}, key=lambda p: str(p), reverse=True)

    rows = []
    for run_dir in run_dirs:
        rows.append({
            "run_name": run_dir.parent.name,
            "run_timestamp": run_dir.name,
            "run_dir": str(run_dir),
            "performance_csv": (run_dir / "performance.csv").exists(),
            "forecast_npy": (run_dir / "forecast.npy").exists(),
            "ensemble_forecast_npy": (run_dir / "ensemble_forecast.npy").exists(),
            "forecasts_arr_npy": (run_dir / "forecasts_arr.npy").exists(),
            "losses_arr_npy": (run_dir / "losses_arr.npy").exists(),
            "checkpoint_manifest_csv": (run_dir / "checkpoint_manifest.csv").exists(),
        })
    latest_runs = pd.DataFrame(rows)
    if latest_runs.empty:
        note("No run directories found yet.", "warn")
    else:
        latest_runs["_run_dt"] = pd.to_datetime(latest_runs["run_timestamp"], format="%Y%m%d_%H%M%S", errors="coerce")
        latest_runs = latest_runs.sort_values(["_run_dt", "run_dir"], ascending=[False, False]).drop(columns="_run_dt")
        display(latest_runs.head(50))

## Forecast and Loss Array Preview

Selects the latest available run by default, loads forecast/loss arrays, and plots actual-vs-forecast when an actual/target array is available in the run or artifacts tree. If actuals are not synced, it still previews forecasts and reports what is missing.

In [ ]:
def choose_run_dir():
    if SELECTED_RUN_DIR:
        path = Path(SELECTED_RUN_DIR)
        return path if path.exists() else None
    candidates = latest_runs.copy() if "latest_runs" in globals() and not latest_runs.empty else pd.DataFrame()
    if candidates.empty:
        return None
    if SELECTED_RUN_NAME:
        candidates = candidates[candidates["run_name"].astype(str).eq(str(SELECTED_RUN_NAME))]
    if candidates.empty:
        return None
    for _, row in candidates.iterrows():
        path = Path(row["run_dir"])
        if (path / "forecast.npy").exists() or (path / "ensemble_forecast.npy").exists() or (path / "forecasts_arr.npy").exists():
            return path
    return Path(candidates.iloc[0]["run_dir"])


def load_array(path):
    path = Path(path)
    if not path.exists():
        return None
    try:
        return np.load(path, allow_pickle=False)
    except ValueError:
        return np.load(path, allow_pickle=True)
    except Exception as exc:
        note(f"Could not load `{path}`: `{type(exc).__name__}: {exc}`", "error")
        return None


def find_actual_array(run_dir):
    names = [
        "actual.npy", "actuals.npy", "y_true.npy", "y_actual.npy", "target.npy", "targets.npy", "y_all.npy", "observed.npy",
    ]
    roots = [run_dir, run_dir.parent, ARTIFACTS_ROOT]
    for root in roots:
        for name in names:
            candidate = root / name
            if candidate.exists():
                arr = load_array(candidate)
                if arr is not None:
                    return candidate, arr
    for pattern in ["*actual*.npy", "*target*.npy", "*y_true*.npy", "*y_all*.npy"]:
        matches = sorted(ARTIFACTS_ROOT.rglob(pattern)) if ARTIFACTS_ROOT.exists() else []
        for candidate in matches[:20]:
            arr = load_array(candidate)
            if arr is not None:
                return candidate, arr
    return None, None


def maturity_index(perf_df, forecast):
    if forecast is None or forecast.ndim < 2:
        return 0
    n_outputs = forecast.shape[1]
    if SELECTED_MATURITY is None or perf_df.empty or "maturity" not in perf_df.columns:
        return 0
    matches = perf_df.index[perf_df["maturity"].astype(str).eq(str(SELECTED_MATURITY))].tolist()
    if matches and 0 <= matches[0] < n_outputs:
        return matches[0]
    return 0


run_dir = choose_run_dir()
if run_dir is None:
    note("No run with forecast arrays is available yet. Sync results first or set `SELECTED_RUN_DIR` above.", "warn")
else:
    note(f"Selected run: `{run_dir}`")
    perf_df = read_csv(run_dir / "performance.csv")
    if not perf_df.empty:
        display(perf_df)

    forecast = load_array(run_dir / "forecast.npy")
    ensemble = load_array(run_dir / "ensemble_forecast.npy")
    forecasts = load_array(run_dir / "forecasts_arr.npy")
    losses = load_array(run_dir / "losses_arr.npy")

    shape_rows = []
    for name, arr in [("forecast", forecast), ("ensemble_forecast", ensemble), ("forecasts_arr", forecasts), ("losses_arr", losses)]:
        if arr is not None:
            shape_rows.append({"array": name, "shape": arr.shape, "dtype": str(arr.dtype), "nan_count": int(np.isnan(arr).sum()) if np.issubdtype(arr.dtype, np.number) else "n/a"})
    if shape_rows:
        display(pd.DataFrame(shape_rows))
    else:
        note("No forecast/loss `.npy` arrays found in the selected run.", "warn")

    forecast_for_plot = ensemble if ensemble is not None else forecast
    if forecast_for_plot is None and forecasts is not None:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            forecast_for_plot = np.nanmean(forecasts, axis=1) if forecasts.ndim >= 3 else forecasts

    actual_path, actual = find_actual_array(run_dir)

    if forecast_for_plot is None:
        note("No forecast array is available to plot.", "warn")
    else:
        try:
            import matplotlib.pyplot as plt

            forecast_2d = np.asarray(forecast_for_plot)
            if forecast_2d.ndim == 1:
                forecast_2d = forecast_2d[:, None]
            out_idx = maturity_index(perf_df, forecast_2d)
            out_idx = min(max(out_idx, 0), forecast_2d.shape[1] - 1)
            yhat = forecast_2d[:, out_idx].astype(float)

            n = min(len(yhat), MAX_PREVIEW_POINTS)
            x = np.arange(n)
            fig, ax = plt.subplots(figsize=(11, 4.5))

            if actual is not None:
                actual_2d = np.asarray(actual)
                if actual_2d.ndim == 1:
                    actual_2d = actual_2d[:, None]
                if actual_2d.shape[0] >= len(yhat):
                    y = actual_2d[-len(yhat):, min(out_idx, actual_2d.shape[1] - 1)].astype(float)
                    ax.plot(x, y[:n], label=f"actual ({actual_path.name})", color="#111827", linewidth=1.5)
                    ax.plot(x, yhat[:n], label="forecast", color="#2563eb", linewidth=1.2)
                    ax.set_title(f"Actual vs forecast preview, output column {out_idx}")
                else:
                    note(f"Found `{actual_path}`, but shape {actual_2d.shape} is shorter than forecast shape {forecast_2d.shape}; plotting forecast only.", "warn")
                    ax.plot(x, yhat[:n], label="forecast", color="#2563eb", linewidth=1.2)
                    ax.set_title(f"Forecast preview, output column {out_idx}")
            else:
                note("No actual/target `.npy` array was found, so actual-vs-forecast is not available. Plotting forecast only.", "warn")
                ax.plot(x, yhat[:n], label="forecast", color="#2563eb", linewidth=1.2)
                ax.set_title(f"Forecast preview, output column {out_idx}")

            ax.axhline(0, color="#d1d5db", linewidth=0.8)
            ax.set_xlabel("Preview step")
            ax.set_ylabel("Value")
            ax.legend(loc="best")
            fig.tight_layout()
            plt.show()

            if losses is not None and np.issubdtype(losses.dtype, np.number):
                loss_arr = np.asarray(losses)
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore", category=RuntimeWarning)
                    loss_series = np.nanmean(loss_arr, axis=tuple(range(1, loss_arr.ndim))) if loss_arr.ndim > 1 else loss_arr
                fig, ax = plt.subplots(figsize=(11, 3.5))
                ax.plot(loss_series[:MAX_PREVIEW_POINTS], color="#dc2626", linewidth=1.2)
                ax.set_title("Mean validation loss preview")
                ax.set_xlabel("Preview step")
                ax.set_ylabel("Loss")
                fig.tight_layout()
                plt.show()
        except ImportError:
            note("`matplotlib` is not installed, so array plots are skipped.", "warn")